In [1]:
import pandas as pd
from pathlib import Path

In [11]:
data_dir = Path("/projects/immunestatus/rheum/airr_format")

metadata = pd.read_csv(data_dir / "metadata.tsv", sep='\t')

In [3]:
tcremp_dir = Path("/projects/immunestatus/rheum/tcremp")

In [11]:
healthy_metadata = metadata[metadata.disease_status == 'hd']

In [12]:
healthy_metadata.b27.value_counts()

b27
neg    21
pos    12
Name: count, dtype: int64

In [13]:
healthy_metadata = healthy_metadata[healthy_metadata.b27 == 'pos'].reset_index(drop=True)
healthy_metadata

,donor_id,disease_status,b27,proj,sample_type,time_point,sample_name,fraction
0,Koc,hd,pos,GCSF,PBMC,normal blood,Koc,NaN
1,Make,hd,pos,ch45,CD45RAdepleted PBMC,after G-CSFstim,Make,NaN
2,p144852,hd,pos,ch45-2,PBMC,after G-CSFstim,p144852,NaN
3,p1004,hd,pos,child_Leu,PBMC,after G-CSFstim,p1004,NaN
4,p112635,hd,pos,ch45-2,PBMC,after G-CSFstim,p112635,NaN
5,LK,hd,pos,NaN,PBMC,NaN,hd_LK_PB_F,bulk
6,LN,hd,pos,NaN,PBMC,NaN,hd_LN_PB_F,bulk
7,OB,hd,pos,NaN,PBMC,NaN,hd_OB_PB_F,bulk
8,Sass,hd,pos,NaN,PBMC,NaN,hd_Sass_PB_F,bulk
9,TwHM2,hd,pos,NaN,PBMC,NaN,hd_TwHM2_PB_F,bulk


In [23]:
ill_metadata = metadata[(metadata.disease_status == 'as') & (~metadata.sample_name.str.contains('CD8'))]

In [14]:
def process(sample):
    print(f'started {sample}')
    df = pd.read_csv(tcremp_dir / f'{sample}_tcremp.tsv', sep='\t', skiprows=lambda i: i % 2 != 0)
    emb_df = df.drop(columns=['clone_id', 'cdr3aa_beta', 'v_beta', 'j_beta'])
    desc_df = df[['cdr3aa_beta', 'v_beta', 'j_beta']].rename(columns={'cdr3aa_beta': 'junction_aa', 
                                                                 'v_beta': 'v_call', 
                                                                 'j_beta': 'j_call'})
    desc_df['count'] = 1
    desc_df['locus'] = 'beta'
    print(f'finished {sample}')
    return desc_df, emb_df

In [15]:
import pandas as pd
import numpy as np
import multiprocessing

samples_desc = []
samples_emb = []

with multiprocessing.Pool(15) as p:
    res = p.map(process, healthy_metadata.sample_name)

for i1, i2 in res:
    samples_desc.append(i1)
    samples_emb.append(i2)

    # Объединяем и сохраняем
pd.concat(samples_desc).to_csv(data_dir / 'joint_hd_b27pos.tsv', sep='\t', index=False)
pd.concat(samples_emb).to_parquet(tcremp_dir / 'joint_hd_embeddings_b27pos.parquet')


started Kocstarted p144852started Makestarted p1004started p112635started hd_LK_PB_Fstarted hd_LN_PB_Fstarted hd_OB_PB_Fstarted hd_Sass_PB_F
started hd_ZN_PB_Fstarted hd_TwHM2_PB_F

started hd_Zbot_PB_F








finished p112635
finished p144852
finished Make
finished Koc
finished hd_OB_PB_F
finished hd_Zbot_PB_F
finished hd_TwHM2_PB_F
finished hd_ZN_PB_F
finished hd_LK_PB_F
finished hd_LN_PB_F
finished hd_Sass_PB_F
finished p1004


In [7]:
import sys
sys.path.append('/home/evlasova/mirpy')
from mir.common.clonotype_dataset import ClonotypeDataset
from mir.common.clonotype import ClonotypeAA

In [12]:
df = pd.read_csv(data_dir / 'joint_hd_b27pos.tsv', sep='\t')


In [13]:
as_seqs = [
    'CASSVGLFSTDTQYF',
    'CASSVGLYSTDTQYF',
    'CASSAGLFSTDTQYF',
    'CASSAGLYSTDTQYF',
    'CASSLGLFSTDTQYF',
    'CASSLGLYSTDTQYF',
    'CASSPGLFSTDTQYF',
    'CASSPGLYSTDTQYF'
]
as_clonotypes = [ClonotypeAA(cdr3aa=x) for x in as_seqs]
as_clonotypes = [ClonotypeAA(cdr3aa=x) for x in as_seqs]
vdjdb = ClonotypeDataset(as_clonotypes)

In [19]:
def has_vdjdb_match(x, threshold=1):
    return len(vdjdb.get_matching_clonotypes(x, threshold=threshold))

In [20]:
df['as'] = df['junction_aa'].apply(has_vdjdb_match)

In [21]:
df[df['as'] > 0]

,junction_aa,v_call,j_call,count,locus,as
6248,CASSLGLASTDTQYF,TRBV5-1*01,TRBJ2-3*01,1,beta,2
24177,CASSLGRFSTDTQYF,TRBV7-3*01,TRBJ2-3*01,1,beta,1
24187,CASSLGQYSTDTQYF,TRBV5-8*01,TRBJ2-3*01,1,beta,1
25169,CASSPGLASTDTQYF,TRBV7-2*01,TRBJ2-3*01,1,beta,2
28001,CASSLGLESTDTQYF,TRBV10-2*01,TRBJ2-3*01,1,beta,2
...,...,...,...,...,...,...
816844,CASSSGLFSTDTQYF,TRBV6-2*01,TRBJ2-3*01,1,beta,4
817501,CASSPGLGSTDTQYF,TRBV11-3*01,TRBJ2-3*01,1,beta,2
818654,CASSLGLDSTDTQYF,TRBV7-9*01,TRBJ2-3*01,1,beta,2
820086,CASSPGSFSTDTQYF,TRBV6-5*01,TRBJ2-3*01,1,beta,1


In [17]:
sum([len(x) for x in samples_desc])

857708

In [8]:
samples = []
for sample in healthy_metadata.sample_name[:20]:
    print(sample)
    samples.append(pd.read_csv(data_dir / f'{sample}.tsv', sep='\t'))
pd.concat(samples).to_csv(data_dir / f'joint_hd.tsv', sep='\t', index=False)
del samples

Koc
Make
p144852
p1004
p112635
p434
p556
p744
p754
p1005
p1321
p1694
p1772
p1841
p1974
p1988
p2048
p2301
TA_pre0
DL_0


In [9]:
samples = []
for sample in ill_metadata.sample_name[:20]:
    print(sample)
    samples.append(pd.read_csv(data_dir / f'{sample}.tsv', sep='\t'))
pd.concat(samples).to_csv(data_dir / f'joint_as.tsv', sep='\t', index=False)
del samples

as_Abd_PB_F
as_Abr_PB_F
as_Ash-110_PB_F_p0
as_Ash-111_PB_F_p0
as_Bal_PB_F
as_Bel_PB_F
as_Bost_PB_F
as_Chaad_PB_F
as_Dv_PB_F
as_Evst_PB_F
as_GE_PB_F
as_Gar_PB_F
as_Gonch_PB_F
as_Kal_PB_F
as_Kud_PB_F
as_Luk_PB_F
as_Mart_PB_F
as_Mikh_PB_F
as_Shep_PB_F
as_Tsib_PB_F


In [6]:
samples = []
for sample in healthy_metadata.sample_name[:20]:
    print(sample)
    samples.append(pd.read_parquet(tcremp_dir / f'{sample}_embeddings.parquet'))
pd.concat(samples).to_parquet(tcremp_dir / f'joint_hd_embeddings.parquet')
del samples

Koc
Make
p144852
p1004
p112635
p434
p556
p744
p754
p1005
p1321
p1694
p1772
p1841
p1974
p1988
p2048
p2301
TA_pre0
DL_0


In [7]:
samples = []
for sample in ill_metadata.sample_name[:20]:
    print(sample)
    samples.append(pd.read_parquet(tcremp_dir / f'{sample}_embeddings.parquet'))
pd.concat(samples).to_parquet(tcremp_dir / f'joint_as_embeddings.parquet')
del samples

as_Abd_PB_F
as_Abr_PB_F
as_Ash-110_PB_F_p0
as_Ash-111_PB_F_p0
as_Bal_PB_F
as_Bel_PB_F
as_Bost_PB_F
as_Chaad_PB_F
as_Dv_PB_F
as_Evst_PB_F
as_GE_PB_F
as_Gar_PB_F
as_Gonch_PB_F
as_Kal_PB_F
as_Kud_PB_F
as_Luk_PB_F
as_Mart_PB_F
as_Mikh_PB_F
as_Shep_PB_F
as_Tsib_PB_F


In [24]:
list(ill_metadata.sample_name)

['as_Abd_PB_F',
 'as_Abr_PB_F',
 'as_Ash-110_PB_F_p0',
 'as_Ash-111_PB_F_p0',
 'as_Bal_PB_F',
 'as_Bel_PB_F',
 'as_Bost_PB_F',
 'as_Chaad_PB_F',
 'as_Dv_PB_F',
 'as_Evst_PB_F',
 'as_GE_PB_F',
 'as_Gar_PB_F',
 'as_Gonch_PB_F',
 'as_Kal_PB_F',
 'as_Kud_PB_F',
 'as_Luk_PB_F',
 'as_Mart_PB_F',
 'as_Mikh_PB_F',
 'as_Shep_PB_F',
 'as_Tsib_PB_F',
 'as_TwHM1_PB_F',
 'as_Uv_PB_F',
 'as_Vas_PB_F',
 'as_Vats_PB_F',
 'as_Vol_PB_F',
 'as_Zakh_PB_F',
 'as_i70_PB_F']

In [26]:
import sys
sys.path.append('/home/evlasova/mirpy')
from mir.common.clonotype_dataset import ClonotypeDataset
from mir.common.clonotype import ClonotypeAA

In [27]:
as_seqs = [
    'CASSVGLFSTDTQYF',
    'CASSVGLYSTDTQYF',
    'CASSAGLFSTDTQYF',
    'CASSAGLYSTDTQYF',
    'CASSLGLFSTDTQYF',
    'CASSLGLYSTDTQYF',
    'CASSPGLFSTDTQYF',
    'CASSPGLYSTDTQYF'
]
as_clonotypes = [ClonotypeAA(cdr3aa=x) for x in as_seqs]

In [28]:
as_clonotypes = [ClonotypeAA(cdr3aa=x) for x in as_seqs]
vdjdb = ClonotypeDataset(as_clonotypes)

In [34]:
def has_vdjdb_match(x, threshold=0):
    return len(vdjdb.get_matching_clonotypes(x, threshold=threshold)) > 0

In [35]:
s_to_df = {}
for s in ill_metadata.sample_name:
    print(s)
    df = pd.read_csv(data_dir / f'{s}.tsv', sep='\t')
    s_to_df[s] = df[df.junction_aa.apply(has_vdjdb_match)]

as_Abd_PB_F
as_Abr_PB_F
as_Ash-110_PB_F_p0
as_Ash-111_PB_F_p0
as_Bal_PB_F
as_Bel_PB_F
as_Bost_PB_F
as_Chaad_PB_F
as_Dv_PB_F
as_Evst_PB_F
as_GE_PB_F
as_Gar_PB_F
as_Gonch_PB_F
as_Kal_PB_F
as_Kud_PB_F
as_Luk_PB_F
as_Mart_PB_F
as_Mikh_PB_F
as_Shep_PB_F
as_Tsib_PB_F
as_TwHM1_PB_F
as_Uv_PB_F
as_Vas_PB_F
as_Vats_PB_F
as_Vol_PB_F
as_Zakh_PB_F
as_i70_PB_F


In [39]:
for s in ill_metadata.sample_name:
    print(s)
    print(s_to_df[s])
    print()

as_Abd_PB_F
Empty DataFrame
Columns: [count, junction_aa, v_call, j_call, locus]
Index: []

as_Abr_PB_F
       count      junction_aa v_call   j_call locus
27705      1  CASSVGLYSTDTQYF  TRBV9  TRBJ2-3  beta

as_Ash-110_PB_F_p0
        count      junction_aa   v_call   j_call locus
772         9  CASSLGLFSTDTQYF  TRBV5-5  TRBJ2-3  beta
106578      1  CASSAGLYSTDTQYF  TRBV5-4  TRBJ2-3  beta

as_Ash-111_PB_F_p0
Empty DataFrame
Columns: [count, junction_aa, v_call, j_call, locus]
Index: []

as_Bal_PB_F
        count      junction_aa   v_call   j_call locus
3471        8  CASSVGLYSTDTQYF    TRBV9  TRBJ2-3  beta
6294        5  CASSVGLYSTDTQYF    TRBV9  TRBJ2-3  beta
103425      1  CASSLGLFSTDTQYF  TRBV5-5  TRBJ2-3  beta

as_Bel_PB_F
       count      junction_aa v_call   j_call locus
1984       5  CASSVGLYSTDTQYF  TRBV9  TRBJ2-3  beta
11048      2  CASSVGLFSTDTQYF  TRBV9  TRBJ2-3  beta

as_Bost_PB_F
        count      junction_aa   v_call   j_call locus
3350        7  CASSVGLFSTDTQYF    TRB